### Установка библиотек

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix, classification_report

from datasets import load_dataset
import numpy as np
import pandas as pd
import random
import json

from transformers import BertConfig, BertModel, BertTokenizerFast, BertForSequenceClassification, TrainingArguments, Trainer, get_cosine_schedule_with_warmup

In [ ]:
def set_random_seed(seed):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

set_random_seed(224)

### Загрузка датасета для анализа тональности

In [ ]:
ds = load_dataset("ai-forever/kinopoisk-sentiment-classification")

### Токенизатор (и модель DeepPavlov)

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained('DeepPavlov/rubert-base-cased')

In [ ]:
def parse_dataset(filepath):
    texts, slots, classes = [], [], []
    with open(filepath, 'r', encoding='utf-8') as file:
        current_text, current_slots, current_classes = [], [], []
        text_id = 0 
        for line in file:
            line = line.strip()
            if line.startswith('# sent_id'):
              sent_text = line.split()[3]
              sent, text = sent_text.split('_')
              if text_id != int(text):
                texts.append(current_text)
                slots.append(current_slots)
                classes.append(current_classes)
                text_id = int(text)
                current_text, current_slots, current_classes = [], [], []
            elif line.startswith('# text'):
              continue
            elif not line:
              continue
            else:
                token = line.split('\t')
                current_text.append(token[1])
                current_classes.append(token[-1])
                current_slots.append(token[-2])
        if current_text:
          texts.append(current_text)
          slots.append(current_slots)
          classes.append(current_classes)
    return texts, slots, classes

In [ ]:
from collections import defaultdict

def extract_classes_and_slots(filepath1):
    classes = set()
    slots = set()
    with open(filepath1, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            fields = line.split('\t')
            if len(fields) < 11:
                continue
            class_field = fields[11]
            slot_field = fields[10]
            classes.add(class_field)
            slots.add(slot_field)
    return classes, slots

def classes_slots_dicts(filepath1, filepath2):
    train_sets = extract_classes_and_slots(filepath1)
    val_sets = extract_classes_and_slots(filepath2)
    classes = set(train_sets[0]).union(set(val_sets[0]))
    slots = set(train_sets[1]).union(set(val_sets[1]))
    print(classes)
    print(slots)

    classes = sorted(classes)
    slots = sorted(slots)
    classes.append('PAD')
    slots.append('PAD')

    class2idx = {cls: idx for idx, cls in enumerate(classes)}
    idx2class = {idx: cls for cls, idx in class2idx.items()}

    slot2idx = {slot: idx for idx, slot in enumerate(slots)}
    idx2slot = {idx: slot for slot, idx in slot2idx.items()}

    return {
        'classes': classes,
        'slots': slots,
        'class2idx': class2idx,
        'idx2class': idx2class,
        'slot2idx': slot2idx,
        'idx2slot': idx2slot
    }

### Кастомный датасет

In [ ]:
class SemDataset(Dataset):
    def __init__(self, texts, slots, classes, labels, tokenizer, slot2id, class2id, max_length):
        self.texts = texts
        self.slots = slots
        self.classes = classes
        self.labels = labels
        self.tokenizer = tokenizer
        self.slot2id = slot2id
        self.class2id = class2id
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        if isinstance(idx, slice):
            return self.slice_token(idx)
        elif isinstance(idx, int):
            return self.get_instance(idx)

    def align_tokens_and_labels(self, tokens, slots, classes):
        word_ids = self.tokenizer.convert_tokens_to_ids(tokens)
        aligned_slots = []
        aligned_classes = []

        current_word_idx = 0
        for word in word_ids:
            current_word = tokens[current_word_idx]
            original_word_token_count = len(self.tokenizer.tokenize(current_word))

            for subtoken in range(original_word_token_count):
                aligned_slots.append(slots[current_word_idx])
                aligned_classes.append(classes[current_word_idx])
            current_word_idx += 1
        return aligned_slots, aligned_classes
    
    def get_instance(self, index):
        tokens = self.texts[index]
        classes = self.classes[index]
        slots = self.slots[index]
        label = self.labels[index]

        encoding = self.tokenizer(
                    tokens,
                    is_split_into_words=True,
                    padding='max_length',
                    truncation=True,
                    max_length=self.max_length,
                    return_tensors='pt'
                )

        slots, classes = self.align_tokens_and_labels(tokens, slots, classes)

        pad_len = self.max_length - len(slots)
        if pad_len > 0:
            for i in range(pad_len):
                slots.append('PAD')
                classes.append('PAD')
        else:
            slots = slots[:self.max_length]
            classes = classes[:self.max_length]

        slots = [self.slot2id[slot] for slot in slots]
        classes = [self.class2id[cls] for cls in classes]

        encoding["slots"] = slots
        encoding["classes"] = classes
        encoding["label"] = label
        encoding["input_ids"] = torch.squeeze(encoding["input_ids"], 0)
        encoding["token_type_ids"] = torch.squeeze(encoding["token_type_ids"], 0)
        encoding["attention_mask"] = torch.squeeze(encoding["attention_mask"], 0)

        return {key: torch.tensor(val) for key, val in encoding.items()}

    def slice_token(self, index):
        start, stop, step = index.indices(len(self.texts))
        result = []
        for index in range(start, stop, step):
            item = self.get_instance(self, index)
            result.append({key: torch.tensor(val) for key, val in item.items()})
        return result

### Параметры

In [ ]:
batch_size = 16
max_length = 512
epochs = 3
lstm_hidden_size=64
semantic_emb_dim=128

In [ ]:
train_dataset_raw = 'sentiment_train_pred.conllu'
val_dataset_raw = 'sentiment_val_pred.conllu'
test_dataset_raw = 'sentiment_test_pred.conllu'
sem_labels_dict = classes_slots_dicts(train_dataset_raw, val_dataset_raw)

train_labels = ds['train']['label']
val_labels = ds['validation']['label']
test_labels = ds['test']['label']

In [ ]:
def create_loader(path_to_dataset, labels, tokenizer, slots2id, classes2id, max_length):
    texts, slots, classes = parse_dataset(path_to_dataset)
    dataset = SemDataset(texts, slots, classes, labels, tokenizer, slots2id, classes2id, max_length)
    return DataLoader(dataset, batch_size, shuffle=False)

In [ ]:
train_loader = create_loader(train_dataset_raw, train_labels, tokenizer, sem_labels_dict['slot2idx'], sem_labels_dict['class2idx'], max_length=max_length)
val_loader = create_loader(val_dataset_raw, val_labels, tokenizer, sem_labels_dict['slot2idx'], sem_labels_dict['class2idx'], max_length=max_length)
test_loader = create_loader(test_dataset_raw, test_labels, tokenizer, sem_labels_dict['slot2idx'], sem_labels_dict['class2idx'], max_length=max_length)

In [ ]:
class BiLSTMPooling(nn.Module):
    def __init__(self, emb_dim, hidden_size):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_size,
            bidirectional=True,
            batch_first=True
        )

    def forward(self, token_embeddings):
        lstm_out, _ = self.lstm(token_embeddings)
        return lstm_out.mean(dim=1)

In [ ]:
class SentimentClassifier(nn.Module):
  def __init__(self, num_labels, num_semantic_classes, num_semantic_slots, semantic_emb_dim, lstm_hidden_size):
    super().__init__()
    self.bert = BertModel.from_pretrained("DeepPavlov/rubert-base-cased") # вынести
    for param in self.bert.parameters():
      if not param.data.is_contiguous():
          param.data = param.data.contiguous()
    self.drop = nn.Dropout(p=0.3) # добавить или убрать
    self.semantic_class_embedding = nn.Embedding(num_semantic_classes, semantic_emb_dim)
    self.semantic_slot_embedding = nn.Embedding(num_semantic_slots, semantic_emb_dim)
    self.semantic_lstm = BiLSTMPooling(emb_dim=semantic_emb_dim, hidden_size=lstm_hidden_size)
    self.classifier = nn.Linear(self.bert.config.hidden_size + 2 * 2 * lstm_hidden_size, num_labels)

  def forward(self, input_ids=None, token_type_ids=None, attention_mask=None, slots=None, classes=None, labels=None):
    _, pooled_output = self.bert(
      input_ids=input_ids,
      attention_mask=attention_mask,
      return_dict=False)

    mask = attention_mask.unsqueeze(-1).float()
    semantic_class_embeds = self.semantic_class_embedding(classes)  
    semantic_slot_embeds = self.semantic_slot_embedding(slots)    
    semantic_class_embeds = semantic_class_embeds * mask
    semantic_slot_embeds = semantic_slot_embeds * mask
    class_summary = self.semantic_lstm(semantic_class_embeds) 
    slot_summary = self.semantic_lstm(semantic_slot_embeds)   
    combined = torch.cat([pooled_output, class_summary, slot_summary], dim=1)
    logits = self.classifier(combined)
    if labels is not None:
        loss_fct = nn.CrossEntropyLoss()
        loss = loss_fct(logits.view(-1, self.classifier.out_features), labels.view(-1))
        return {"loss": loss, "logits": logits}
    else:
        return {"logits": logits}
    
  def save_pretrained(self):
    state_dict = self.state_dict()
    for key, value in state_dict.items():
        if not value.is_contiguous():
            state_dict[key] = value.contiguous()
    torch.save(state_dict, "./pytorch_model.bin")

In [ ]:
num_labels = len(set(ds['train']['label']))
num_semantic_classes = len(sem_labels_dict['classes'])
num_semantic_slots = len(sem_labels_dict['slots'])

model = SentimentClassifier(num_labels=num_labels,
    num_semantic_classes=num_semantic_classes,
    num_semantic_slots=num_semantic_slots,
    semantic_emb_dim=semantic_emb_dim,
    lstm_hidden_size=lstm_hidden_size)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

### Функция для подсчета метрик

In [ ]:
def compute_metrics(p):

    predictions, labels = p

    predictions = predictions.argmax(axis=-1)
    cm = confusion_matrix(labels, predictions)

    print("Confusion Matrix:\n", cm)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

### Кросс-валидация

#### Аргументы на кросс-валидации

In [ ]:
training_args = TrainingArguments(
    output_dir="model/checkpoints",
    eval_strategy="epoch",
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=epochs,
    logging_dir="outputs/logs",
    logging_steps=10,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

#### Функция для кросс-валидации

In [ ]:
def hp_space_fn(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True),
        "weight_decay" : trial.suggest_float("weight_decay", 1e-5, 0.1, log=True)
        }

def model_init():
    return BertForSequenceClassification.from_pretrained('DeepPavlov/rubert-base-cased', num_labels=num_labels)

#### Дефолтный трейнер из transformers

In [ ]:
trainer = Trainer(
    model=model,
    model_init = model_init,
    args=training_args,
    train_dataset=train_loader.dataset,
    eval_dataset=val_loader.dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    )


#### Обучение с кросс-валидацией

In [ ]:
best_run = trainer.hyperparameter_search(
    hp_space=hp_space_fn,
    n_trials=5,
    direction="maximize",
    backend="optuna"
)

#### Результаты

In [ ]:
best_params = best_run.hyperparameters

### Обучение с оптимизатором и шедулером

In [ ]:
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()))
total_steps = len(train_loader.dataset) * epochs * batch_size

scheduler = get_cosine_schedule_with_warmup(
  optimizer,
  num_warmup_steps=total_steps*0.05,
  num_training_steps=total_steps
)

learning_rate = 2e-5
weight_decay = 0.01

In [ ]:
training_args = TrainingArguments(
    output_dir="model/checkpoint",
    eval_strategy="epoch",
    learning_rate = learning_rate,
    weight_decay = weight_decay,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=epochs,
    logging_dir="outputs/logs",
    logging_steps=10,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    gradient_checkpointing=True
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_loader.dataset,
    eval_dataset=val_loader.dataset,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, scheduler)
    )

In [ ]:
train_metrics = trainer.train().metrics

with open("outputs/metrics/train_metrics.json", "w") as f:
    json.dump(train_metrics, f, indent=2)

eval_results = trainer.evaluate()
print(f"Evaluation Results: {eval_results}")

with open("outputs/metrics/eval_metrics.json", "w") as f:
    json.dump(eval_results, f, indent=2)

trainer.save_model("model/final_model")
tokenizer.save_pretrained("tokenizer/final_tokenizer")

In [ ]:
test_results = trainer.evaluate(eval_dataset=test_loader.dataset, metric_key_prefix="test")
print(f"Evaluation Results: {test_results}")

with open("outputs/metrics/eval_metrics.json", "w") as f:
    json.dump(test_results, f, indent=2)

### Тестирование модели

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
def test_model(model, data_loader, device):
  model = model.eval()

  all_preds = torch.tensor([], device=device)
  all_trues = torch.tensor([], device=device)

  with torch.no_grad():
    for d in data_loader:
      input_ids = d["input_ids"].to(device)
      attention_mask = d["attention_mask"].to(device)
      targets = d["labels"].to(device)

      outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
      )
      preds = torch.argmax(outputs['logits'], axis=-1)
      all_preds = torch.cat((all_preds, preds), -1)
      all_trues = torch.cat((all_trues, targets), -1)

  precision, recall, f1, _ = precision_recall_fscore_support(all_trues.cpu(), all_preds.cpu(), average='macro')
  acc = accuracy_score(all_trues.cpu(), all_preds.cpu())
  return {
      'accuracy': acc,
      'f1': f1,
      'precision': precision,
      'recall': recall
  }

### Результаты на тестовом датасете

In [ ]:
test = test_model(model, test_loader, device)
test